Import libraries

In [1]:
import numpy as np
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.model_selection import GridSearchCV

Load data

In [3]:
df = pd.read_csv("C:\Machine_Learning_Projects\energy-load-forecasting\smart-household-energy-forecasting\data\processed\household_features.csv")

df.head()

,Datetime,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3,Month,Day,Weekday,Hour,lag_1,lag_60,lag_1440,rolling_60,rolling_1440
0,2006-12-17 17:24:00,3.390,0.170,232.65,14.6,0.0,0.0,17.0,12,17,6,17,3.238,4.864,4.216,3.527967,2.487154
1,2006-12-17 17:25:00,3.360,0.164,231.20,14.4,0.0,0.0,16.0,12,17,6,17,3.390,4.868,5.360,3.502833,2.485765
2,2006-12-17 17:26:00,3.400,0.162,232.86,14.6,0.0,0.0,17.0,12,17,6,17,3.360,4.876,5.374,3.478233,2.484394
3,2006-12-17 17:27:00,3.396,0.164,232.89,14.6,0.0,0.0,17.0,12,17,6,17,3.400,4.810,5.388,3.454667,2.483011
4,2006-12-17 17:28:00,3.282,0.100,232.33,14.0,0.0,0.0,16.0,12,17,6,17,3.396,4.790,3.666,3.429533,2.482744


Drop null values

In [4]:
# Drop missing values
df = df.dropna()

print(df.shape)

(1950981, 17)


Split Data for training and testing

In [5]:
X = df.drop(['Datetime', "Global_active_power"], axis = 1)
y = df["Global_active_power"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(1560784, 15) (390197, 15)


Model Comparison

In [13]:
models = {
    "LinearRegression": LinearRegression(),
    "RandomForest": RandomForestRegressor(
        n_estimators = 100,
        max_depth = 10,
        min_samples_split = 5,
        min_samples_leaf = 2, 
        n_jobs = -1,
        random_state = 0
        ),
    "DecisionTree": DecisionTreeRegressor(
        max_depth = 10,
        min_samples_split = 5,
        min_samples_leaf = 2
    )
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    results[name] = mae

    print(f"{name} MAE: {mae}")


Training LinearRegression...
LinearRegression MAE: 0.025363174015759413

Training RandomForest...
RandomForest MAE: 0.01638177982870318

Training DecisionTree...
DecisionTree MAE: 0.017621639545291584


Hyper Parameter tuning

In [ ]:
param_grid = {
    "n_estimators" : [10, 50, 100],
    "max_depth" : [5, 10, None],
    "min_samples_split": [2, 5]
}

grid = GridSearchCV(
    RandomForestRegressor(random_state = 0),
    param_grid = param_grid,
    cv = 3,
    verbose = 1
)

grid.fit(X_train, y_train)

print('Best Parameters: ', grid.best_params_)

Fitting 3 folds for each of 18 candidates, totalling 54 fits
